# RFT-0002 — R1 native self-RFT A/B BF16 LoRA on A100

This notebook trains two controlled LoRA lanes from the same immutable
`r1_native_verified_full.csv` corpus. Both lanes start from a fresh
`Qwen/Qwen2.5-3B-Instruct`; no adapter is continued from a prior lane.

- **Lane A (legend control):** r32 / alpha64 / lr 5e-5
- **Lane B (low-drift):** r16 / alpha32 / lr 3e-5
- Common: BF16 LoRA, seq 2048, one epoch, effective batch 48,
  assistant-only loss, cosine, warmup 3%, no packing.

The notebook never reads leaderboard or official-test data. It verifies
the immutable R1 SHA-256, performs an actual forward/backward A100 batch
probe using the longest examples, then uses the same selected micro-batch
for both lanes. Checkpoints and reports are written to Google Drive and
interrupted training resumes from the latest checkpoint.

Run Cell 1 once. If this runtime previously imported Transformers or PEFT,
restart the runtime once after Cell 1, then continue from Cell 2.

In [1]:
# Cell 1 — Install one tested, mutually compatible training stack.
# Do not reinstall torch: Colab's preinstalled torch matches its CUDA runtime.
%pip install -q --no-cache-dir \
  "transformers==4.52.3" "peft==0.16.0" "accelerate==1.7.0" \
  "datasets==3.6.0" "bitsandbytes==0.46.0" \
  "sentencepiece>=0.2.0" "tensorboard~=2.19.0" \
  "wandb>=0.19,<1" "protobuf<6"

print("[SETUP] Installation complete. Restart once only if these packages were already imported.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.2/40.2 kB 129.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 169.6 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.3/472.3 kB 375.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 362.1/362.1 kB 395.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 398.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 222.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 274.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 399.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 309.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.24.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.

In [3]:
# Cell 2 — Imports, deterministic runtime, Drive mount, and Unicode-safe paths.
import gc
import hashlib
import inspect
import json
import math
import os
import platform
import re
import subprocess
import time
import traceback
import unicodedata
from pathlib import Path

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["WANDB_MODE"] = "offline"
os.environ["WANDB_PROJECT"] = "deep-learning-challenge-2026"

import numpy as np
import pandas as pd
import torch
from google.colab import drive

BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
MODEL_REVISION = "main"
RUN_ID = "RFT-0002-r1-ab-bf16-lora-a100"
SEED = 20260824
EXPECTED_R1_SHA256 = "5d5fd3f2051264596bf05aefabc44b88724351510f43c4f623d36029a708101e"
EXPECTED_R1_ROWS = 17495
EXPECTED_R1_QUESTIONS = 9764
MAX_SEQ_LENGTH = 2048
EFFECTIVE_BATCH_SIZE = 48

SYSTEM_PROMPT = "You are a helpful assistant that solves math problems step by step."
USER_SUFFIX = (
    "Solve this step by step, then give the final answer as a single integer "
    "inside \\boxed{}."
)
PROMPT_VERSION = "champion_r1_boxed_k4_v1"

LANES = {
    "lane_a_legend_control": {"r": 32, "alpha": 64, "learning_rate": 5e-5},
    "lane_b_low_drift": {"r": 16, "alpha": 32, "learning_rate": 3e-5},
}

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()

def compact_name(value):
    value = unicodedata.normalize("NFC", str(value)).casefold()
    return re.sub(r"[\s_\-]+", "", value)

MOUNT_ROOT = Path("/content/drive")
if not (MOUNT_ROOT / "MyDrive").exists():
    drive.mount(str(MOUNT_ROOT))
DRIVE_ROOT = MOUNT_ROOT / "MyDrive"

project_candidates = [
    path for path in DRIVE_ROOT.iterdir()
    if path.is_dir() and compact_name(path.name) == compact_name("2026소중한챌린지")
]
assert project_candidates, "Google Drive 최상위에서 2026소중한챌린지 폴더를 찾지 못했습니다."

r1_candidates = []
for project in project_candidates:
    direct = project / "runs" / "RFT-0002-r1-native-k4-full" / "data" / "r1_native_verified_full.csv"
    if direct.exists():
        r1_candidates.append(direct)
    else:
        r1_candidates.extend(project.glob("runs/RFT-0002-r1-native-k4-full/data/r1_native_verified_full.csv"))

r1_candidates = list(dict.fromkeys(r1_candidates))
matching = [path for path in r1_candidates if sha256_file(path) == EXPECTED_R1_SHA256]
assert len(matching) == 1, (
    f"Expected exactly one immutable R1 file with SHA {EXPECTED_R1_SHA256}; "
    f"found {len(matching)}. Candidates={list(map(str, r1_candidates))}"
)

R1_DATA_PATH = matching[0]
SOURCE_RUN_DIR = R1_DATA_PATH.parents[1]
PROJECT_DIR = SOURCE_RUN_DIR.parents[1]
SOURCE_MANIFEST_PATH = SOURCE_RUN_DIR / "reports" / "r1_native_manifest.json"
assert SOURCE_MANIFEST_PATH.exists(), SOURCE_MANIFEST_PATH

EXP_DIR = PROJECT_DIR / "runs" / RUN_ID
DATA_DIR = EXP_DIR / "data"
REPORT_DIR = EXP_DIR / "reports"
LOG_DIR = EXP_DIR / "logs"
TB_DIR = LOG_DIR / "tensorboard"
WANDB_DIR = LOG_DIR / "wandb"
for path in [DATA_DIR, REPORT_DIR, TB_DIR, WANDB_DIR]:
    path.mkdir(parents=True, exist_ok=True)
os.environ["WANDB_DIR"] = str(WANDB_DIR)

READ_PATHS = [R1_DATA_PATH, SOURCE_MANIFEST_PATH]
forbidden = [
    path for path in READ_PATHS
    if any(term in path.name.casefold() for term in ["leaderboard", "submission"])
    or path.name.casefold() in {"deep_chal_math_test.csv", "deep_chal_math_dataset_test.csv"}
]
assert not forbidden, f"Forbidden evaluation input read: {forbidden}"

assert torch.cuda.is_available(), "A100 GPU runtime required."
gpu_name = torch.cuda.get_device_name(0)
total_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
assert "A100" in gpu_name.upper() and total_gib >= 39, (gpu_name, total_gib)
assert torch.cuda.is_bf16_supported(), "BF16-capable A100 required."

print("[PATH] R1 data:", R1_DATA_PATH)
print("[PATH] experiment:", EXP_DIR)
print("[GPU]", gpu_name, f"{total_gib:.1f} GiB")
print("[SHA]", sha256_file(R1_DATA_PATH))

Mounted at /content/drive
[PATH] R1 data: /content/drive/MyDrive/2026소중한챌린지/runs/RFT-0002-r1-native-k4-full/data/r1_native_verified_full.csv
[PATH] experiment: /content/drive/MyDrive/2026소중한챌린지/runs/RFT-0002-r1-ab-bf16-lora-a100
[GPU] NVIDIA A100-SXM4-40GB 39.5 GiB
[SHA] 5d5fd3f2051264596bf05aefabc44b88724351510f43c4f623d36029a708101e


In [4]:
# Cell 3 — Load, verify, and tokenize the immutable R1 corpus with zero truncation.
from datasets import Dataset
from transformers import AutoTokenizer, set_seed

set_seed(SEED)
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL, revision=MODEL_REVISION, use_fast=True, token=False
)
tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token
tokenizer.padding_side = "right"

r1 = pd.read_csv(R1_DATA_PATH, dtype=str, keep_default_na=False)
r1.columns = [column.strip() for column in r1.columns]
required = {"id", "question", "answer", "solution", "source", "origin_shard"}
assert required.issubset(r1.columns), (required, r1.columns.tolist())
assert len(r1) == EXPECTED_R1_ROWS, len(r1)
assert r1["id"].nunique() == EXPECTED_R1_QUESTIONS
assert r1["answer"].str.fullmatch(r"-?\d+").all()
assert r1.groupby("id").size().max() <= 2

TERMINAL_BOX_RE = re.compile(r"\\boxed\s*\{\s*(-?\d(?:[\d,]*\d)?)\s*\}")

def terminal_boxed(text):
    matches = list(TERMINAL_BOX_RE.finditer(str(text or "")))
    if not matches:
        return None
    match = matches[-1]
    tail = str(text)[match.end():].strip()
    while True:
        previous = tail
        tail = re.sub(r"^(?:\\\)|\\\]|\$\$|\$)", "", tail).strip()
        if tail == previous:
            break
    tail = re.sub(r"^[.!]+$", "", tail).strip()
    return str(int(match.group(1).replace(",", ""))) if not tail else None

assert all(terminal_boxed(solution) == answer for solution, answer in zip(r1["solution"], r1["answer"]))

def user_content(question):
    return f"{str(question).strip()}\n\n{USER_SUFFIX}"

def prompt_text(question):
    return tokenizer.apply_chat_template(
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_content(question)},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )

def full_text(question, solution):
    return tokenizer.apply_chat_template(
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_content(question)},
            {"role": "assistant", "content": str(solution).strip()},
        ],
        tokenize=False,
        add_generation_prompt=False,
    )

def tokenize_row(example):
    prompt_ids = tokenizer(prompt_text(example["question"]), add_special_tokens=False)["input_ids"]
    encoded = tokenizer(full_text(example["question"], example["solution"]), add_special_tokens=False)
    input_ids = encoded["input_ids"]
    assert input_ids[:len(prompt_ids)] == prompt_ids, f"Assistant boundary mismatch: {example['id']}"
    labels = [-100] * len(prompt_ids) + input_ids[len(prompt_ids):]
    return {
        "input_ids": input_ids,
        "attention_mask": encoded["attention_mask"],
        "labels": labels,
        "total_tokens": len(input_ids),
        "assistant_tokens": sum(label != -100 for label in labels),
    }

raw_dataset = Dataset.from_pandas(
    r1[["id", "question", "answer", "solution"]], preserve_index=False
)
tokenized = raw_dataset.map(tokenize_row, desc="Tokenize R1", num_proc=2)
token_lengths = np.asarray(tokenized["total_tokens"])
assistant_lengths = np.asarray(tokenized["assistant_tokens"])

eligible_mask = token_lengths <= MAX_SEQ_LENGTH
rejected_count = int((~eligible_mask).sum())
rejected_fraction = rejected_count / len(tokenized)
assert rejected_fraction <= 0.02, (
    f"Actual chat-template token rejection {rejected_fraction:.2%} exceeds 2%; stop and audit."
)

eligible_indices = np.flatnonzero(eligible_mask).tolist()
rejected_indices = np.flatnonzero(~eligible_mask).tolist()
eligible = tokenized.select(eligible_indices)
token_reject = r1.iloc[rejected_indices].copy()
if rejected_indices:
    token_reject["total_tokens"] = token_lengths[~eligible_mask]
TOKEN_REJECT_PATH = DATA_DIR / "r1_token_length_reject.csv"
token_reject.to_csv(TOKEN_REJECT_PATH, index=False, encoding="utf-8")

# Diagnostic eval-loss split is grouped by question ID. It is not a promotion metric.
eligible_ids = sorted(set(eligible["id"]))
eval_ranked_ids = sorted(
    eligible_ids,
    key=lambda qid: hashlib.sha256(f"{SEED}|{qid}".encode()).hexdigest(),
)
diagnostic_eval_ids = set(eval_ranked_ids[:min(256, len(eval_ranked_ids) // 20)])
train_indices = [i for i, qid in enumerate(eligible["id"]) if qid not in diagnostic_eval_ids]
eval_indices = [i for i, qid in enumerate(eligible["id"]) if qid in diagnostic_eval_ids]
train_dataset = eligible.select(train_indices).remove_columns(
    ["id", "question", "answer", "solution", "total_tokens", "assistant_tokens"]
)
eval_dataset = eligible.select(eval_indices).remove_columns(
    ["id", "question", "answer", "solution", "total_tokens", "assistant_tokens"]
)

assert train_dataset.num_rows + eval_dataset.num_rows == eligible.num_rows
assert set(eligible.select(train_indices)["id"]).isdisjoint(set(eligible.select(eval_indices)["id"]))
assert max(len(row) for row in train_dataset["input_ids"]) <= MAX_SEQ_LENGTH
assert all(any(label != -100 for label in labels) for labels in train_dataset["labels"])

TRAIN_IDS_PATH = DATA_DIR / "r1_train_ids.csv"
DIAG_EVAL_IDS_PATH = DATA_DIR / "r1_diagnostic_eval_ids.csv"
pd.DataFrame({"id": sorted(set(eligible.select(train_indices)["id"]))}).to_csv(
    TRAIN_IDS_PATH, index=False
)
pd.DataFrame({"id": sorted(diagnostic_eval_ids)}).to_csv(DIAG_EVAL_IDS_PATH, index=False)

length_report = {
    "source_rows": len(r1),
    "eligible_rows": eligible.num_rows,
    "rejected_rows": rejected_count,
    "rejected_fraction": rejected_fraction,
    "train_rows": train_dataset.num_rows,
    "diagnostic_eval_rows": eval_dataset.num_rows,
    "total_token_quantiles": {
        str(q): float(np.quantile(token_lengths, q)) for q in [0, .5, .9, .95, .99, 1]
    },
    "assistant_token_quantiles": {
        str(q): float(np.quantile(assistant_lengths, q)) for q in [0, .5, .9, .95, .99, 1]
    },
    "actual_truncation": 0,
}
(REPORT_DIR / "token_length_report.json").write_text(
    json.dumps(length_report, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("[DATA] eligible/train/diag-eval:", eligible.num_rows, train_dataset.num_rows, eval_dataset.num_rows)
print("[DATA] token rejects:", rejected_count)
print("[DATA] max eligible tokens:", max(len(row) for row in train_dataset["input_ids"]))
print(json.dumps(length_report["total_token_quantiles"], indent=2))

tokenizer_config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenize R1 (num_proc=2):   0%|          | 0/17495 [00:00<?, ? examples/s]

Exception ignored in: <_io.BytesIO object at 0x7ad8359d61b0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/tblib/pickling_support.py", line 13, in unpickle_traceback
    def unpickle_traceback(tb_frame, tb_lineno, tb_next):
BufferError: Existing exports of data: object cannot be re-sized
Exception ignored in: <_io.BytesIO object at 0x7ad8359d6520>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/tblib/pickling_support.py", line 13, in unpickle_traceback
    def unpickle_traceback(tb_frame, tb_lineno, tb_next):
BufferError: Existing exports of data: object cannot be re-sized
Exception ignored in: <_io.BytesIO object at 0x7ad89537c9f0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/tblib/pickling_support.py", line 13, in unpickle_traceback
    def unpickle_traceback(tb_frame, tb_lineno, tb_next):
BufferError: Existing exports of data: object cannot be re-sized
Exception ignored in: 

[DATA] eligible/train/diag-eval: 17495 17039 456
[DATA] token rejects: 0
[DATA] max eligible tokens: 1425
{
  "0": 135.0,
  "0.5": 414.0,
  "0.9": 633.0,
  "0.95": 717.0,
  "0.99": 897.0600000000013,
  "1": 1425.0
}


In [5]:
# Cell 4 — Actual A100 forward/backward batch probe on the longest examples.
# Uses r32 (the larger lane) and a conservative torch AdamW optimizer.
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, DataCollatorForSeq2Seq

TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]
BATCH_CANDIDATES = [(24, 2), (16, 3), (12, 4), (8, 6)]
MIN_FREE_GIB = 4.5

def release_cuda(*objects):
    for obj in objects:
        try:
            del obj
        except Exception:
            pass
    gc.collect()
    torch.cuda.empty_cache()

def load_fresh_lora(rank, alpha):
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        revision=MODEL_REVISION,
        torch_dtype=torch.bfloat16,
        device_map={"": 0},
        token=False,
    )
    model.config.use_cache = False
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.enable_input_require_grads()
    model = get_peft_model(model, LoraConfig(
        r=rank,
        lora_alpha=alpha,
        lora_dropout=0.0,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=TARGET_MODULES,
    ))
    model.train()
    return model

longest_indices = sorted(
    range(train_dataset.num_rows),
    key=lambda i: len(train_dataset[i]["input_ids"]),
    reverse=True,
)[:max(micro for micro, _ in BATCH_CANDIDATES)]
longest_examples = [train_dataset[i] for i in longest_indices]

probe_results = []
selected_micro = selected_accum = None

for micro, accum in BATCH_CANDIDATES:
    model = optimizer = batch = outputs = None
    status, error_text = "failed", ""
    peak_allocated_gib = peak_reserved_gib = min_free_gib = None
    try:
        release_cuda()
        torch.cuda.reset_peak_memory_stats()
        model = load_fresh_lora(rank=32, alpha=64)
        collator = DataCollatorForSeq2Seq(
            tokenizer=tokenizer,
            model=model,
            padding=True,
            label_pad_token_id=-100,
            pad_to_multiple_of=8,
        )
        batch = collator(longest_examples[:micro])
        batch = {key: value.to("cuda:0") for key, value in batch.items()}
        optimizer = torch.optim.AdamW(
            [parameter for parameter in model.parameters() if parameter.requires_grad],
            lr=5e-5,
        )
        optimizer.zero_grad(set_to_none=True)
        outputs = model(**batch)
        outputs.loss.backward()
        optimizer.step()
        torch.cuda.synchronize()

        peak_allocated_gib = torch.cuda.max_memory_allocated() / 1024**3
        peak_reserved_gib = torch.cuda.max_memory_reserved() / 1024**3
        free_bytes, _ = torch.cuda.mem_get_info()
        min_free_gib = free_bytes / 1024**3
        status = "pass" if min_free_gib >= MIN_FREE_GIB else "insufficient_headroom"
    except (torch.cuda.OutOfMemoryError, RuntimeError) as exc:
        error_text = repr(exc)
        if "out of memory" not in error_text.casefold() and not isinstance(exc, torch.cuda.OutOfMemoryError):
            raise
        status = "oom"
    finally:
        if optimizer is not None:
            optimizer.zero_grad(set_to_none=True)
        del outputs, batch, optimizer, model
        gc.collect()
        torch.cuda.empty_cache()

    result = {
        "micro_batch": micro,
        "gradient_accumulation": accum,
        "effective_batch": micro * accum,
        "status": status,
        "peak_allocated_gib": peak_allocated_gib,
        "peak_reserved_gib": peak_reserved_gib,
        "free_after_step_gib": min_free_gib,
        "error": error_text[-1000:],
    }
    probe_results.append(result)
    print("[PROBE]", json.dumps(result, ensure_ascii=False), flush=True)
    if status == "pass":
        selected_micro, selected_accum = micro, accum
        break

assert selected_micro is not None, "No batch candidate retained at least 4.5 GiB headroom."
assert selected_micro * selected_accum == EFFECTIVE_BATCH_SIZE

BATCH_PROBE_REPORT = REPORT_DIR / "a100_batch_probe.json"
BATCH_PROBE_REPORT.write_text(json.dumps({
    "gpu": gpu_name,
    "total_gib": total_gib,
    "minimum_free_gib": MIN_FREE_GIB,
    "tested_on_longest_examples": len(longest_examples),
    "results": probe_results,
    "selected": {
        "per_device_train_batch_size": selected_micro,
        "gradient_accumulation_steps": selected_accum,
        "effective_batch_size": EFFECTIVE_BATCH_SIZE,
    },
}, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"[PROBE SELECTED] micro={selected_micro} accum={selected_accum} effective=48")
print("[PROBE REPORT]", BATCH_PROBE_REPORT)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

...

[PROBE] {"micro_batch": 16, "gradient_accumulation": 3, "effective_batch": 48, "status": "oom", "peak_allocated_gib": null, "peak_reserved_gib": null, "free_after_step_gib": null, "error": "OutOfMemoryError('CUDA out of memory. Tried to allocate 12.97 GiB. GPU 0 has a total capacity of 39.49 GiB of which 10.00 GiB is free. Including non-PyTorch memory, this process has 29.49 GiB memory in use. Of the allocated memory 28.90 GiB is allocated by PyTorch, and 86.44 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)')"}


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

[PROBE] {"micro_batch": 12, "gradient_accumulation": 4, "effective_batch": 48, "status": "oom", "peak_allocated_gib": null, "peak_reserved_gib": null, "free_after_step_gib": null, "error": "OutOfMemoryError('CUDA out of memory. Tried to allocate 9.73 GiB. GPU 0 has a total capacity of 39.49 GiB of which 6.07 GiB is free. Including non-PyTorch memory, this process has 33.42 GiB memory in use. Of the allocated memory 32.83 GiB is allocated by PyTorch, and 86.50 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)')"}


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

[PROBE] {"micro_batch": 8, "gradient_accumulation": 6, "effective_batch": 48, "status": "pass", "peak_allocated_gib": 30.366702556610107, "peak_reserved_gib": 31.59375, "free_after_step_gib": 7.38226318359375, "error": ""}
[PROBE SELECTED] micro=8 accum=6 effective=48
[PROBE REPORT] /content/drive/MyDrive/2026소중한챌린지/runs/RFT-0002-r1-ab-bf16-lora-a100/reports/a100_batch_probe.json


In [6]:
# Cell 5 — Shared resumable Trainer implementation for both independent lanes.
from peft import LoraConfig, get_peft_model
from transformers import DataCollatorForSeq2Seq, Trainer, TrainingArguments
from transformers.trainer_utils import get_last_checkpoint

class MemoryLoggingTrainer(Trainer):
    def log(self, logs, start_time=None):
        logs = dict(logs)
        if torch.cuda.is_available():
            logs["gpu/memory_allocated_gib"] = round(torch.cuda.memory_allocated() / 1024**3, 3)
            logs["gpu/memory_reserved_gib"] = round(torch.cuda.memory_reserved() / 1024**3, 3)
            logs["gpu/max_memory_allocated_gib"] = round(torch.cuda.max_memory_allocated() / 1024**3, 3)
            try:
                free_bytes, _ = torch.cuda.mem_get_info()
                logs["gpu/memory_free_gib"] = round(free_bytes / 1024**3, 3)
            except Exception:
                pass
        print("[TRAIN-LOG]", json.dumps(logs, ensure_ascii=False, default=str), flush=True)
        return super().log(logs, start_time)

def json_safe(value):
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, np.generic):
        return value.item()
    return value

def train_lane(lane_name):
    config = dict(LANES[lane_name])
    lane_dir = EXP_DIR / lane_name
    checkpoint_dir = lane_dir / "checkpoints"
    adapter_final = lane_dir / "adapter_final"
    lane_report_dir = lane_dir / "reports"
    lane_tb_dir = TB_DIR / lane_name
    lane_wandb_dir = WANDB_DIR / lane_name
    for path in [checkpoint_dir, lane_report_dir, lane_tb_dir, lane_wandb_dir]:
        path.mkdir(parents=True, exist_ok=True)

    completed_report = lane_report_dir / "training_report.json"
    if (adapter_final / "adapter_config.json").exists() and completed_report.exists():
        print(f"[SKIP] {lane_name} already complete:", adapter_final)
        return json.loads(completed_report.read_text(encoding="utf-8"))

    os.environ["WANDB_DIR"] = str(lane_wandb_dir)
    os.environ["WANDB_RUN_ID"] = f"{RUN_ID}-{lane_name}"
    os.environ["WANDB_RESUME"] = "allow"

    release_cuda()
    model = load_fresh_lora(config["r"], config["alpha"])
    model.print_trainable_parameters()
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    assert 0 < trainable / total < 0.03, (trainable, total)

    collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        model=model,
        padding=True,
        label_pad_token_id=-100,
        pad_to_multiple_of=8,
    )

    optimizer_steps = math.ceil(
        train_dataset.num_rows / (selected_micro * selected_accum)
    )
    quarter_steps = max(1, round(optimizer_steps / 4))
    evaluation_arg = (
        "eval_strategy"
        if "eval_strategy" in inspect.signature(TrainingArguments).parameters
        else "evaluation_strategy"
    )

    kwargs = {
        "output_dir": str(checkpoint_dir),
        "logging_dir": str(lane_tb_dir),
        "num_train_epochs": 1.0,
        "learning_rate": config["learning_rate"],
        "per_device_train_batch_size": selected_micro,
        "per_device_eval_batch_size": max(4, min(selected_micro, 12)),
        "gradient_accumulation_steps": selected_accum,
        "gradient_checkpointing": True,
        "gradient_checkpointing_kwargs": {"use_reentrant": False},
        "optim": "paged_adamw_8bit",
        "lr_scheduler_type": "cosine",
        "warmup_ratio": 0.03,
        "weight_decay": 0.0,
        "max_grad_norm": 1.0,
        "bf16": True,
        "fp16": False,
        "tf32": True,
        "logging_steps": 10,
        "logging_first_step": True,
        "eval_steps": quarter_steps,
        "save_steps": quarter_steps,
        "save_strategy": "steps",
        "save_total_limit": 5,
        "load_best_model_at_end": False,
        "prediction_loss_only": True,
        "report_to": ["tensorboard", "wandb"],
        "run_name": f"{RUN_ID}-{lane_name}",
        "remove_unused_columns": False,
        "group_by_length": True,
        "dataloader_num_workers": 2,
        "dataloader_pin_memory": True,
        "seed": SEED,
        "data_seed": SEED,
    }
    kwargs[evaluation_arg] = "steps"
    for optional, value in {
        "include_tokens_per_second": True,
        "include_num_input_tokens_seen": True,
    }.items():
        if optional in inspect.signature(TrainingArguments).parameters:
            kwargs[optional] = value

    training_config = {
        "run_id": RUN_ID,
        "lane": lane_name,
        "base_model": BASE_MODEL,
        "model_revision": MODEL_REVISION,
        "method": "BF16 LoRA, assistant-only loss, no packing",
        "r": config["r"],
        "alpha": config["alpha"],
        "dropout": 0.0,
        "target_modules": TARGET_MODULES,
        "learning_rate": config["learning_rate"],
        "epochs": 1.0,
        "max_seq_length": MAX_SEQ_LENGTH,
        "per_device_train_batch_size": selected_micro,
        "gradient_accumulation_steps": selected_accum,
        "effective_batch_size": EFFECTIVE_BATCH_SIZE,
        "scheduler": "cosine",
        "warmup_ratio": 0.03,
        "optimizer": "paged_adamw_8bit",
        "weight_decay": 0.0,
        "train_rows": train_dataset.num_rows,
        "diagnostic_eval_rows": eval_dataset.num_rows,
        "quarter_save_steps": quarter_steps,
        "estimated_optimizer_steps": optimizer_steps,
        "prompt_version": PROMPT_VERSION,
        "source_r1_sha256": EXPECTED_R1_SHA256,
        "seed": SEED,
    }
    (lane_report_dir / "training_config.json").write_text(
        json.dumps(training_config, ensure_ascii=False, indent=2), encoding="utf-8"
    )

    trainer = MemoryLoggingTrainer(
        model=model,
        args=TrainingArguments(**kwargs),
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=collator,
    )
    last_checkpoint = get_last_checkpoint(str(checkpoint_dir))
    print(f"[TRAIN] {lane_name} resume={last_checkpoint}", flush=True)
    torch.cuda.reset_peak_memory_stats()
    started = time.time()
    try:
        result = trainer.train(resume_from_checkpoint=last_checkpoint)
    except Exception as exc:
        failure = {
            "status": "failed",
            "lane": lane_name,
            "error": repr(exc),
            "traceback": traceback.format_exc(),
            "runtime_seconds": time.time() - started,
            "training_config": training_config,
        }
        (lane_report_dir / "failure_report.json").write_text(
            json.dumps(failure, ensure_ascii=False, indent=2), encoding="utf-8"
        )
        raise

    runtime_seconds = time.time() - started
    adapter_final.mkdir(parents=True, exist_ok=True)
    trainer.save_model(str(adapter_final))
    tokenizer.save_pretrained(str(adapter_final))
    trainer.state.save_to_json(str(lane_report_dir / "trainer_state.json"))

    checkpoints = sorted(
        str(path) for path in checkpoint_dir.glob("checkpoint-*")
        if (path / "adapter_config.json").exists()
    )
    report = {
        "status": "trained_pending_generation_em",
        "lane": lane_name,
        "training_config": training_config,
        "train_metrics": json_safe(result.metrics),
        "runtime_seconds": runtime_seconds,
        "gpu_max_memory_allocated_gib": torch.cuda.max_memory_allocated() / 1024**3,
        "adapter_final": str(adapter_final),
        "checkpoints": checkpoints,
        "official_evaluation_files_read": [],
    }
    completed_report.write_text(
        json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    print(f"[TRAIN DONE] {lane_name} {runtime_seconds/60:.1f} min")
    print("[ADAPTER]", adapter_final)

    del trainer, model, collator
    gc.collect()
    torch.cuda.empty_cache()
    return report

print("[TRAINER READY] selected micro/accum:", selected_micro, selected_accum)

[TRAINER READY] selected micro/accum: 8 6


In [7]:
# Cell 6 — Train/resume Lane A (legend control).
lane_a_report = train_lane("lane_a_legend_control")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

trainable params: 59,867,136 || all params: 3,145,805,824 || trainable%: 1.9031


No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


[TRAIN] lane_a_legend_control resume=None


wandb: WARNING `resume` will be ignored since W&B syncing is set to `offline`. Starting a new run with run id RFT-0002-r1-ab-bf16-lora-a100-lane_a_legend_control.


Step,Training Loss,Validation Loss
89,0.128500,0.140076
178,0.126400,0.138452
267,0.116600,0.138505


[TRAIN-LOG] {"loss": 0.0938, "grad_norm": 0.0969688892364502, "learning_rate": 0.0, "gpu/memory_allocated_gib": 12.003, "gpu/memory_reserved_gib": 36.824, "gpu/max_memory_allocated_gib": 33.105, "gpu/memory_free_gib": 2.027}
[TRAIN-LOG] {"loss": 0.1209, "grad_norm": 0.2417970597743988, "learning_rate": 4.0909090909090915e-05, "gpu/memory_allocated_gib": 12.003, "gpu/memory_reserved_gib": 36.865, "gpu/max_memory_allocated_gib": 33.105, "gpu/memory_free_gib": 1.986}
[TRAIN-LOG] {"loss": 0.132, "grad_norm": 0.10670475661754608, "learning_rate": 4.9933307091588796e-05, "gpu/memory_allocated_gib": 12.003, "gpu/memory_reserved_gib": 36.869, "gpu/max_memory_allocated_gib": 33.105, "gpu/memory_free_gib": 1.982}
[TRAIN-LOG] {"loss": 0.1343, "grad_norm": 0.1381741166114807, "learning_rate": 4.9662976890711167e-05, "gpu/memory_allocated_gib": 12.003, "gpu/memory_reserved_gib": 36.869, "gpu/max_memory_allocated_gib": 33.105, "gpu/memory_free_gib": 1.982}
[TRAIN-LOG] {"loss": 0.1386, "grad_norm": 0

In [8]:
# Cell 7 — Train/resume Lane B (low-drift candidate) from a fresh base.
lane_b_report = train_lane("lane_b_low_drift")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


[TRAIN] lane_b_low_drift resume=None


Step,Training Loss,Validation Loss
89,0.128500,0.140665
178,0.126000,0.138809
267,0.116500,0.138949


[TRAIN-LOG] {"loss": 0.0938, "grad_norm": 0.06627578288316727, "learning_rate": 0.0, "gpu/memory_allocated_gib": 12.45, "gpu/memory_reserved_gib": 37.271, "gpu/max_memory_allocated_gib": 33.573, "gpu/memory_free_gib": 1.58}
[TRAIN-LOG] {"loss": 0.1203, "grad_norm": 0.0920325368642807, "learning_rate": 2.454545454545455e-05, "gpu/memory_allocated_gib": 12.45, "gpu/memory_reserved_gib": 37.293, "gpu/max_memory_allocated_gib": 33.573, "gpu/memory_free_gib": 1.558}
[TRAIN-LOG] {"loss": 0.1335, "grad_norm": 0.0724613219499588, "learning_rate": 2.9959984254953276e-05, "gpu/memory_allocated_gib": 12.45, "gpu/memory_reserved_gib": 37.293, "gpu/max_memory_allocated_gib": 33.573, "gpu/memory_free_gib": 1.558}
[TRAIN-LOG] {"loss": 0.1345, "grad_norm": 0.08367228507995605, "learning_rate": 2.97977861344267e-05, "gpu/memory_allocated_gib": 12.45, "gpu/memory_reserved_gib": 37.293, "gpu/max_memory_allocated_gib": 33.573, "gpu/memory_free_gib": 1.558}
[TRAIN-LOG] {"loss": 0.1386, "grad_norm": 0.09655

In [12]:
# Cell 8 — Persist the immutable A/B training summary.
import transformers
import peft
import datasets

source_manifest = json.loads(SOURCE_MANIFEST_PATH.read_text(encoding="utf-8"))
git_state = {"commit": None, "dirty": None}
try:
    git_state["commit"] = subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=str(PROJECT_DIR), text=True,
        stderr=subprocess.DEVNULL,
    ).strip()
    git_state["dirty"] = bool(subprocess.check_output(
        ["git", "status", "--porcelain"], cwd=str(PROJECT_DIR), text=True,
        stderr=subprocess.DEVNULL,
    ).strip())
except Exception:
    pass

final_report = {
    "experiment_id": RUN_ID,
    "status": "trained_pending_tune_generation_em",
    "hypothesis": (
        "On the identical native self-RFT R1 corpus, lower LoRA rank and LR "
        "preserve more raw-Qwen math ability than the legend-control recipe."
    ),
    "base_model": BASE_MODEL,
    "model_revision": MODEL_REVISION,
    "source": {
        "path": str(R1_DATA_PATH),
        "sha256": sha256_file(R1_DATA_PATH),
        "rows": len(r1),
        "questions": r1["id"].nunique(),
        "source_manifest": str(SOURCE_MANIFEST_PATH),
        "source_manifest_sha256": sha256_file(SOURCE_MANIFEST_PATH),
        "source_summary": source_manifest,
    },
    "token_length_report": length_report,
    "batch_probe": json.loads(BATCH_PROBE_REPORT.read_text(encoding="utf-8")),
    "lanes": {"lane_a": lane_a_report, "lane_b": lane_b_report},
    "promotion_metric": "generation integer Exact Match on fixed tune, then one-shot dev",
    "promotion_status": "pending",
    "gpu": {"name": gpu_name, "total_gib": total_gib},
    "versions": {
        "python": platform.python_version(),
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "peft": peft.__version__,
        "datasets": datasets.__version__,
    },
    "git": git_state,
    "official_evaluation_files_read": [],
}
FINAL_REPORT_JSON = REPORT_DIR / "training_ab_report.json"
FINAL_REPORT_MD = REPORT_DIR / "training_ab_report.md"
FINAL_REPORT_JSON.write_text(
    json.dumps(final_report, ensure_ascii=False, indent=2, default=str), encoding="utf-8"
)
FINAL_REPORT_MD.write_text(
    "\n".join([
        f"# {RUN_ID}",
        "",
        f"- Source rows/questions: {len(r1)} / {r1['id'].nunique()}",
        f"- Source SHA-256: `{sha256_file(R1_DATA_PATH)}`",
        f"- Batch: micro {selected_micro} × accum {selected_accum} = 48",
        f"- Lane A: r32 / alpha64 / lr 5e-5 — `{lane_a_report['adapter_final']}`",
        f"- Lane B: r16 / alpha32 / lr 3e-5 — `{lane_b_report['adapter_final']}`",
        "- Status: trained; generation Exact Match evaluation pending",
        "",
    ]),
    encoding="utf-8",
)
print("[REPORT]", FINAL_REPORT_JSON)
print("[REPORT]", FINAL_REPORT_MD)
print("[NEXT] Do not select by loss. Run checkpoint generation EM on tune, then one-shot dev.")

[REPORT] /content/drive/MyDrive/2026소중한챌린지/runs/RFT-0002-r1-ab-bf16-lora-a100/reports/training_ab_report.json
[REPORT] /content/drive/MyDrive/2026소중한챌린지/runs/RFT-0002-r1-ab-bf16-lora-a100/reports/training_ab_report.md
[NEXT] Do not select by loss. Run checkpoint generation EM on tune, then one-shot dev.


In [13]:
# Cell 9 — Optional TensorBoard. In VS Code, use the printed log directory if widgets do not render.
print("TensorBoard logs:", TB_DIR)
# In the Colab browser UI, uncomment both lines:
# %load_ext tensorboard
# %tensorboard --logdir "$TB_DIR"

TensorBoard logs: /content/drive/MyDrive/2026소중한챌린지/runs/RFT-0002-r1-ab-bf16-lora-a100/logs/tensorboard


In [15]:
# Cell 10 — Optional runtime release. Disabled by default.
DISCONNECT_RUNTIME = True
if DISCONNECT_RUNTIME:
    from google.colab import runtime
    runtime.unassign()
else:
    print("[RUNTIME] retained. Set DISCONNECT_RUNTIME=True only after reports are safely written.")